# Understanding Global Climate Change
This project analyzes global temperature trends from 2000 to 2015 and builds interactive visualizations to explore patterns, uncertainties, and regional differences.

In [1]:
%pip install pandas numpy plotly dash

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: C:\Users\srira\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
%pip install jupyter_dash

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: C:\Users\srira\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
%pip install dash_bootstrap_components

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: C:\Users\srira\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objs as go
from dash import dcc, html, Input, Output
from jupyter_dash import JupyterDash
import dash_bootstrap_components as dbc


## Data Loading

In [5]:
# Load the dataset
df = pd.read_csv("data/GlobalLandTemperaturesByCountry.csv")
df['dt'] = pd.to_datetime(df['dt'])
df['Year'] = df['dt'].dt.year
df = df[df['Year'] >= 2000]
df.head()


,dt,AverageTemperature,AverageTemperatureUncertainty,Country,Year
3074,2000-01-01,0.197,0.407,Åland,2000
3075,2000-02-01,-0.023,0.399,Åland,2000
3076,2000-03-01,0.615,0.429,Åland,2000
3077,2000-04-01,4.124,0.348,Åland,2000
3078,2000-05-01,8.557,0.447,Åland,2000


## Data Cleaning

In [6]:
# Rename columns for ease
df = df.rename(columns={
    'AverageTemperature': 'AvgTemp',
    'AverageTemperatureUncertainty': 'TempUncertainty'
})

# Drop missing values
df = df.dropna(subset=['AvgTemp', 'Country'])

# Group by Country-Year
df_grouped = df.groupby(['Country', 'Year'])[['AvgTemp', 'TempUncertainty']].mean().reset_index()

df_grouped.head()


,Country,Year,AvgTemp,TempUncertainty
0,Afghanistan,2000,15.497833,0.500667
1,Afghanistan,2001,15.778083,0.539083
2,Afghanistan,2002,15.537667,0.445167
3,Afghanistan,2003,14.916000,0.456667
4,Afghanistan,2004,15.770917,0.482500


## Feature Engineering

In [7]:
# Create rolling temperature averages
df_grouped['RollingTemp'] = df_grouped.groupby('Country')['AvgTemp'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())


## Static Visualizations

In [8]:
# Line plot
fig1 = px.line(df_grouped[df_grouped['Country']=='India'], x='Year', y='AvgTemp', title='Avg Temperature Over Time (India)', template='plotly_dark')
fig1.show(renderer="browser")

# Scatter plot
fig2 = px.scatter(df_grouped, x='AvgTemp', y='TempUncertainty', color='Country', title='Temp vs Uncertainty', template='plotly_dark')
fig2.show(renderer="browser")

# Box plot
fig3 = px.box(df_grouped, x='Country', y='AvgTemp', title='Temperature Distribution per Country', template='plotly_dark')
fig3.show(renderer="browser")

# Bubble chart
fig4 = px.scatter(df_grouped, x='Year', y='AvgTemp', size='TempUncertainty', color='Country', title='Bubble Chart: Temp vs Uncertainty', template='plotly_dark')
fig4.show(renderer="browser")

# Histogram
fig5 = px.histogram(df_grouped, x='AvgTemp', nbins=50, title='Temperature Frequency Distribution', template='plotly_dark')
fig5.show(renderer="browser")

# Choropleth map
fig6 = px.choropleth(df_grouped, locations="Country", locationmode="country names", color="AvgTemp", hover_name="Country", animation_frame="Year", color_continuous_scale=px.colors.sequential.Plasma, title='World Map Avg Temperature')
fig6.show(renderer="browser")


## Interactive Dash Dashboard

In [9]:
# Building JupyterDash App
app = JupyterDash(__name__, external_stylesheets=[dbc.themes.CYBORG])

app.layout = dbc.Container([
    html.H1("Global Climate Change Dashboard", style={'textAlign': 'center', 'marginTop': 20}),
    dbc.Row([
        dbc.Col([
            html.Label("Select Country:"),
            dcc.Dropdown(
                id='country-dropdown',
                options=[{'label': i, 'value': i} for i in df_grouped['Country'].unique()],
                value='India'
            ),
            html.Label("Select Year Range:"),
            dcc.RangeSlider(
                id='year-slider',
                min=df_grouped['Year'].min(),
                max=df_grouped['Year'].max(),
                value=[2000, 2015],
                marks={str(year): str(year) for year in range(2000, 2016, 3)}
            ),
            html.Label("Temperature Range:"),
            dcc.RangeSlider(
                id='temp-slider',
                min=-20,
                max=40,
                value=[0, 30],
                marks={i: f"{i}°C" for i in range(-20, 41, 10)}
            )
        ], width=3),
        dbc.Col([
            dcc.Graph(id='line-plot'),
            dcc.Graph(id='scatter-plot'),
            dcc.Graph(id='box-plot'),
            dcc.Graph(id='histogram-plot'),
            dcc.Graph(id='choropleth-plot')
        ], width=9)
    ])
], fluid=True)

# Callback to update all plots
@app.callback(
    Output('line-plot', 'figure'),
    Output('scatter-plot', 'figure'),
    Output('box-plot', 'figure'),
    Output('histogram-plot', 'figure'),
    Output('choropleth-plot', 'figure'),
    Input('country-dropdown', 'value'),
    Input('year-slider', 'value'),
    Input('temp-slider', 'value')
)
def update_graphs(selected_country, selected_years, temp_range):
    filtered = df_grouped[(df_grouped['Year'] >= selected_years[0]) &
                          (df_grouped['Year'] <= selected_years[1]) &
                          (df_grouped['AvgTemp'] >= temp_range[0]) &
                          (df_grouped['AvgTemp'] <= temp_range[1])]
    
    # Line Chart
    line_fig = px.line(filtered[filtered['Country'] == selected_country], x='Year', y='AvgTemp',
                       title=f'Average Temperature Over Time: {selected_country}', template='plotly_dark')
    
    # Scatter Plot
    scatter_fig = px.scatter(filtered[filtered['Country'] == selected_country], x='Year', y='TempUncertainty',
                             color='AvgTemp', title='Year vs Temp Uncertainty', template='plotly_dark')
    
    # Box Plot
    box_fig = px.box(filtered[filtered['Country'] == selected_country], y='AvgTemp',
                     title='Temperature Distribution', template='plotly_dark')
    
    # Histogram
    hist_fig = px.histogram(filtered[filtered['Country'] == selected_country], x='AvgTemp',
                            nbins=30, title='Temperature Frequency Distribution', template='plotly_dark')
    
    # Choropleth Map
    choropleth_fig = px.choropleth(
        filtered, locations="Country", locationmode="country names",
        color="AvgTemp", hover_name="Country",
        animation_frame="Year", color_continuous_scale=px.colors.sequential.Plasma,
        title='World Map of Avg Temperature'
    )
    
    return line_fig, scatter_fig, box_fig, hist_fig, choropleth_fig

# Run app
app.run(mode='external')
#Navigate to http://127.0.0.1:8050/ after this cell to get full view of dashboard

C:\Users\srira\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\dash\dash.py:587: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.

